# EDA — `fraud-review-queue` (Día 2, **timeboxed a un día**)

> **Reglas del timebox (design.md §10.2 e `instrucciones.md`).**
> 1. Las 5 preguntas de abajo se escribieron **antes** de abrir el notebook. No se
>    añaden más sobre la marcha.
> 2. Cada pregunta existe porque **alimenta una decisión concreta aguas abajo**.
>    Si una figura no responde una de las 5, no va.
> 3. **8 figuras como máximo. Un día. Se responde y se cierra.**
> 4. Las V-columns **no** se estudian una por una hoy (§5.4). Si aparecen, es solo
>    en la Q4, por patrón de nulos.
>
> El riesgo de hoy no es la dificultad: es la **seducción**. Vas a encontrar
> patrones interesantes en las V-columns y vas a querer perseguirlos. No lo hagas.

**Las respuestas las escribes tú.** Yo dejé las preguntas, el *por qué* y el
código de arranque. La interpretación es el entregable, y es tuya.


## Setup

In [ ]:
import duckdb




pd.set_option("display.max_columns", 60)

# Ajusta si corriste ingest.py con otra ruta.
TXN = "data/processed/transactions.parquet"
IDY = "data/processed/identity.parquet"

con = duckdb.connect()

# Días de corte del split (design.md §4.1) — para marcarlos en las figuras.
TRAIN_END, EMBARGO_END, CALIB_END = 119, 129, 155

con.execute(f"SELECT COUNT(*) AS n, AVG(isFraud) AS fraud_rate FROM read_parquet('{TXN}')").df()


## Q1 — ¿Dónde vive el fraude en el eje del **monto**?

**Por qué es LA pregunta.** La magnitud de toda la tesis depende de esto. `V` es
máximo en `p ≈ 0.25` por álgebra (garantizado), pero que la **diferencia en
dólares** entre rankear por valor y rankear por score sea *grande* depende de que
exista fraude de **monto alto**. Si los fraudes de IEEE-CIS son casi todos de
monto bajo, el efecto puede ser trivial.

**Decisión que alimenta:** fija tu expectativa realista **antes** del H6, y te
dice si debes tener presente la contingencia §13.5. Mirar esta figura hoy evita la
tentación de torcer parámetros el día crítico.


In [ ]:
# Distribución de TransactionAmt por clase (log-x), y tasa de fraude por decil de monto.
df = con.execute(f"""
    SELECT TransactionAmt, isFraud
    FROM read_parquet('{TXN}')
""").df()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
bins = np.logspace(0, np.log10(df.TransactionAmt.max()), 60)
ax[0].hist(df.loc[df.isFraud == 0, "TransactionAmt"], bins=bins, alpha=.5, density=True, label="legit")
ax[0].hist(df.loc[df.isFraud == 1, "TransactionAmt"], bins=bins, alpha=.5, density=True, label="fraud")
ax[0].set_xscale("log"); ax[0].set_xlabel("TransactionAmt"); ax[0].set_ylabel("densidad"); ax[0].legend()

df["amt_decile"] = pd.qcut(df.TransactionAmt, 10, labels=False, duplicates="drop")
rate = df.groupby("amt_decile").isFraud.mean()
ax[1].bar(rate.index, rate.values)
ax[1].set_xlabel("decil de monto"); ax[1].set_ylabel("tasa de fraude")
plt.tight_layout()

# TODO (tú): ¿el fraude se concentra en montos bajos o hay masa en montos altos?
# ¿Qué fracción del *dinero* en fraude está en el top-cuartil de monto?


**Respuesta (la escribes tú):**

_..._


## Q2 — ¿El **split temporal** es viable? ¿Hay positivos suficientes en calibración?

**Por qué.** El split (train 0–119 / embargo 120–129 / calib 130–155 / test 156+)
solo funciona si cada partición tiene volumen y —crítico— si la **partición de
calibración tiene positivos suficientes** para ajustar isotónica (§7.3). Si la
tasa de fraude se desploma en los últimos días, el calibrador y el test se quedan
sin señal.

**Decisión que alimenta:** confirma (o corrige) los días de corte de `SplitConfig`
antes de escribir `split.py` mañana. Si calib tiene pocos positivos, Platt sobre
isotónica.


In [ ]:
daily = con.execute(f"""
    SELECT day, COUNT(*) AS n, SUM(isFraud) AS n_fraud, AVG(isFraud) AS rate
    FROM read_parquet('{TXN}')
    GROUP BY day ORDER BY day
""").df()

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(daily.day, daily.n); ax[0].set_ylabel("transacciones/día")
ax[1].plot(daily.day, daily.rate); ax[1].set_ylabel("tasa de fraude/día"); ax[1].set_xlabel("day")
for a in ax:
    for d in (TRAIN_END, EMBARGO_END, CALIB_END):
        a.axvline(d, ls="--", c="k", alpha=.4)
plt.tight_layout()

# Positivos por partición: el número que decide Platt vs. isotónica.
con.execute(f"""
    SELECT
      CASE WHEN day <= {TRAIN_END} THEN '1_train'
           WHEN day <= {EMBARGO_END} THEN '2_embargo'
           WHEN day <= {CALIB_END} THEN '3_calib'
           ELSE '4_test' END AS part,
      COUNT(*) AS n, SUM(isFraud) AS n_fraud, AVG(isFraud) AS rate
    FROM read_parquet('{TXN}')
    GROUP BY part ORDER BY part
""").df()


**Respuesta (la escribes tú):**

_..._


## Q3 — ¿`D1n = day − D1` se comporta como **constante por tarjeta**? (viabilidad del UID)

**Por qué.** Mañana (Día 3) construyes el UID = `card1 + addr1 + D1n` y las
features retrospectivas encima. Todo eso asume que `D1n` (aprox. fecha de registro
de la tarjeta) es estable dentro de una misma tarjeta. Si no lo es, el entity
resolution no sirve y hay que replantear.

**Decisión que alimenta:** de-riesga el Día 3 **hoy**, con una comprobación barata,
en lugar de descubrir el problema a mitad del feature engineering.


In [ ]:
probe = con.execute(f"""
    SELECT
        card1, addr1,
        COUNT(*) AS n,
        COUNT(DISTINCT (day - D1)) AS n_distinct_d1n
    FROM read_parquet('{TXN}')
    WHERE D1 IS NOT NULL AND addr1 IS NOT NULL
    GROUP BY card1, addr1
    HAVING COUNT(*) >= 3
""").df()

# Si D1n fuera perfectamente constante por (card1, addr1), n_distinct_d1n == 1.
share_constant = (probe.n_distinct_d1n == 1).mean()
print(f"Grupos (card1, addr1) con >=3 txns: {len(probe):,}")
print(f"Fracción con D1n único: {share_constant:.1%}")

# TODO (tú): ¿es lo bastante alta para confiar en el UID? ¿Qué haces con la cola
# de grupos con varios D1n — ruido de addr1, o el proxy se rompe?


**Respuesta (la escribes tú):**

_..._


## Q4 — Cobertura de **identity** y patrones de nulos (la única mirada a las V-columns hoy)

**Por qué.** Dos decisiones de feature engineering salen de aquí:
1. `train_identity` solo cubre una fracción de las transacciones → **left join, no
   inner** (§3.2). Necesitas el número exacto.
2. Las 339 V-columns se agrupan en bloques de Vesta que **comparten patrón de
   nulos** (§5.4). Ver cuántos bloques hay decide si eliges una representante por
   bloque o se las dejas enteras a LightGBM.

**Decisión que alimenta:** estrategia de identity (join) y de V-columns. **Nada
de arqueología columna a columna.**


In [ ]:
n_txn = con.execute(f"SELECT COUNT(*) FROM read_parquet('{TXN}')").fetchone()[0]
n_idy = con.execute(f"SELECT COUNT(*) FROM read_parquet('{IDY}')").fetchone()[0]
print(f"Cobertura de identity: {n_idy:,} / {n_txn:,} = {n_idy/n_txn:.1%}  -> left join.")

# Patrones de nulos de las V-columns: cuántos bloques distintos hay.
vcols = [c for c in con.execute(f"SELECT * FROM read_parquet('{TXN}') LIMIT 0").df().columns
         if c.startswith("V")]
sample = con.execute(
    f"SELECT {', '.join(vcols)} FROM read_parquet('{TXN}') USING SAMPLE 20000 ROWS"
).df()
null_signature = sample.isnull().mean().round(3)          # % nulos por V-col
n_blocks = null_signature.nunique()
print(f"V-columns: {len(vcols)}  ->  {n_blocks} patrones de nulos distintos (bloques de Vesta).")

# TODO (tú): ¿cuántos bloques? ¿Una representante por bloque, o todas a LightGBM?


**Respuesta (la escribes tú):**

_..._


## Q5 — ¿Qué categóricas base separan fraude? (`ProductCD`, `card4`, `card6`, email)

**Por qué.** Antes de invertir en features, confirma qué categóricas de baja
cardinalidad tienen señal, y mira la cardinalidad de las altas (`card1`, `addr1`)
para la codificación por frecuencia (§5.1). Barato y directamente accionable.

**Decisión que alimenta:** el conjunto de features base del Día 3, y qué
categóricas merecen frequency encoding calculado **solo sobre train**.


In [ ]:
for colname in ["ProductCD", "card4", "card6"]:
    print(f"=== {colname} ===")
    print(con.execute(f"""
        SELECT {colname}, COUNT(*) AS n, AVG(isFraud) AS fraud_rate
        FROM read_parquet('{TXN}')
        GROUP BY {colname} ORDER BY n DESC
    """).df().to_string(index=False))
    print()

# Cardinalidad de las de alta cardinalidad (candidatas a frequency encoding).
print("=== cardinalidad ===")
print(con.execute(f"""
    SELECT
      COUNT(DISTINCT card1) AS card1,
      COUNT(DISTINCT addr1) AS addr1,
      COUNT(DISTINCT P_emaildomain) AS p_email
    FROM read_parquet('{TXN}')
""").df().to_string(index=False))

# TODO (tú): ¿qué categorías tienen tasa de fraude muy por encima de la base?
# ¿Qué dominios de email agrupas en 'other'?


**Respuesta (la escribes tú):**

_..._


## Cierre del timebox

- [ ] Las 5 preguntas están respondidas en prosa (no solo figuras).
- [ ] ≤ 8 figuras. Ninguna figura que no responda una de las 5.
- [ ] Confirmé los días de corte del split (Q2) o anoté el ajuste.
- [ ] Decidí: Platt vs. isotónica (Q2), UID sí/no (Q3), V-columns estrategia (Q4).
- [ ] **Una pregunta de entrevista** para la bitácora §V (probablemente sale de Q1
      o Q3).
- [ ] Commit: `feat: csv->parquet ingestion` + `docs: timeboxed EDA`.
- [ ] **Cerré el notebook. No sigo explorando las V-columns.**

> Mañana (Día 3): `test_no_future_leakage.py` **primero**, luego el UID.
